In [ ]:
import sys
import os
# Add src directory to path to import kg_voyager module
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))  # Add project root for ABSTAT package

from llm import process_subgraphs_yaml
# load dotenv
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
import yaml
from llm import process_single_subgraph_pipeline, load_prompt_template

with open("subgraphs_output/03-24-13-dblp.yaml") as f:
    subgraphs = yaml.safe_load(f)

sg = subgraphs[9]
template = load_prompt_template()

result = process_single_subgraph_pipeline(
    subgraph=sg,
    subgraph_index=9,
    endpoint="dblp",
    model="anthropic/claude-sonnet-4-6",
    template=template,  
    timeout=70,
    fail_empty=False,
    verify_iterations=2, 
)

print(f"Is meaningful: {result['is_meaningful']}")
if result.get("verification_status"):
    print(f"Verification: {result['verification_status']} (iteration {result['finalized_at_iteration']})")
    print(f"Most meaningful pair index: {result['most_meaningful_index']}")
for qa in result["qa_pairs"]:
    print(f"\nQ: {qa['question']}")
    print(f"SPARQL:\n{qa['sparql_query']}")
    print(f"Exec success: {qa['execution_results']['success']} | Count: {qa['execution_results']['count']}")
    if qa.get("verification_status"):
        print(f"Verified: {qa['verification_status']} — {qa.get('verification_reason', '')}")   

Is meaningful: True
Verification: correct (iteration 1)
Most meaningful pair index: 2

Q: Which identified persons have authored more than one article, and how many articles have they each authored? List them in descending order of article count.
SPARQL:
PREFIX dblp: <https://dblp.org/rdf/schema#>

SELECT ?authorName (COUNT(?article) AS ?articleCount)
WHERE {
  ?article a dblp:Article ;
           dblp:authoredBy ?author .
  ?author a dblp:Person ;
          dblp:creatorname ?authorName .
}
GROUP BY ?authorName
HAVING (COUNT(?article) > 1)
ORDER BY DESC(?articleCount)
Exec success: True | Count: 0
Verified: correct — 

Q: Who are all the authors — both identified persons and ambiguous creators — of the article with DOI https://doi.org/10.1109/ACCESS.2019.2933987?
SPARQL:
PREFIX dblp: <https://dblp.org/rdf/schema#>

SELECT ?authorName
WHERE {
  ?article a dblp:Article ;
           dblp:doi <https://doi.org/10.1109/ACCESS.2019.2933987> ;
           dblp:authoredBy ?author .
  {
    ?auth

In [2]:
# Process YAML: generate questions, execute queries, save to qa_output/
output_path = process_subgraphs_yaml(
    input_yaml_path="subgraphs_output/04-23-21-dream-kg.yaml",
    endpoint="dream-kg",
    model="gemini/gemini-3-pro-preview",
    timeout=140,
    verify_iterations=2, 
    max_workers=10,
    max_subgraphs=150,
    focus_entity_qa=False,
    ents_ratio=0.4,
    introduce_complexity_step=True
)
print(f"Output saved to: {output_path}")

Processing 120 subgraphs [Gen + Exec + Complexity + Verify(×2) + Reword] with 10 workers...


Pipeline Progress (Gen + Exec): 100%|██████████| 120/120 [08:26<00:00,  4.22s/it]



── Verification Summary ──
  After cycle 1: 127/152 (83.6%) pairs correct
  After cycle 2: 146/152 (96.1%) pairs correct
──────────────────────────
Output saved to: subgraphs_output/qa_output/04-23-21-dream-kg-gemini3propreview.yaml


In [4]:
# Convert YAML output to JSONL filtering only successful queries with results
from yaml_to_jsonl import convert_yaml_to_jsonl

# If optimal=True, keep only the most meaningful QA per subgraph, then sample
convert_yaml_to_jsonl(output_path, num_samples=50, optimal=False, endpoint="nasa-gesdisc-kg", allow_empty_results=True, execute=True)

Reading subgraphs_output/qa_output/04-23-21-dream-kg-gemini3propreview.yaml...
Filtering entries...


Processing subgraphs: 100%|██████████| 120/120 [00:38<00:00,  3.13it/s]

Sampling 50 entries from 146 candidates (51 subgraphs)...
Writing 50 entries to subgraphs_output/qa_output/04-23-21-dream-kg-gemini3propreview_sample50.jsonl...
Conversion complete.
Total QA pairs scanned: 152
Valid candidates found: 146
Entries written: 50
Output saved to: subgraphs_output/qa_output/04-23-21-dream-kg-gemini3propreview_sample50.jsonl
